# NOORIX-AI Unified Offline Model Training — Public Kaggle Notebook
**Prepared by Muhammad Farhan · NOORIX-AI**

One-click training controller: discover attached datasets, prepare supported manifests, run/evaluate available specialist models, and export ONE combined ZIP with each model in its own folder. No need to run separate specialist notebooks manually.

NOORIX presents one unified AI engine/API but uses specialist models for different tasks. One neural network is not a reliable replacement for object detection, OCR, currency classification, ASR, and TTS.

Attach datasets using Kaggle Add Input, select a GPU runtime, and choose Run All. Enable Internet when packages or pretrained weights need downloading. First run uses a limited baseline profile. Only trained/exported models are included; skipped tasks are clearly marked.

Final output: NOORIX_AI_ALL_MODELS.zip. Offline inference is possible after model weights, dependencies, and compatible task adapters are installed locally. Windows EXE and Android APK packaging are separate steps.

In [ ]:
from pathlib import Path
import os, sys, json, re, shutil, zipfile, hashlib, subprocess, time, traceback
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO = WORK / 'NOORIX-AI-training-source'
OUT = WORK / 'NOORIX_MASTER_OUTPUT'
OUT.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version)
print('Working directory:', WORK)
try:
    import torch
    print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as e: print('PyTorch check:', e)


## 1. Automatic dataset discovery
The controller searches nested Kaggle inputs, including /kaggle/input/datasets/. It selects one clearly Pakistan-labelled currency dataset and prepares Urdu ASR audio/transcript pairs by filename stem. Duplicate-looking currency sources are not merged automatically. OCR is skipped unless exact image-to-text annotations are configured.

In [ ]:
from pathlib import Pathimport re, jsonimport pandas as pdINPUT = Path('/kaggle/input')OUT = Path('/kaggle/working/NOORIX_MASTER_OUTPUT')OUT.mkdir(parents=True, exist_ok=True)# Known attached PKR datasets. Select one source; do not combine identical-looking sources.pkr_candidates = [    INPUT / 'datasets/muhammadammarabid/pakistanicurrencydataset/data-rescaled',    INPUT / 'datasets/rafayqayyum/pakistan-image-currency-dataset/currencyDataset',]pkr_dir = Nonepkr_classes = {}for candidate in pkr_candidates:    if not candidate.is_dir():        continue    summary = {}    for folder in sorted(candidate.iterdir()):        if folder.is_dir():            count = sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'})            if count:                summary[folder.name] = count    eligible = [k for k in summary if re.search(r'(?<!\\d)(10|20|50|75|100|500|1000|5000)(?!\\d)', k) and ('front' in k.lower() or 'back' in k.lower())]    if len(eligible) >= 4:        pkr_dir, pkr_classes = candidate, summary        breakprint('PKR selected:', pkr_dir)print('PKR classes:', pkr_classes)print('PKR images:', sum(pkr_classes.values()))# Match TSV MP3 stems to actual WAV files recursively.URDU_BASE = INPUT / 'datasets/muhammadahmedansari/urdu-dataset-20000'URDU_TSV = URDU_BASE / 'final_main_dataset.tsv'URDU_AUDIO = URDU_BASE / 'limited_wav_files'URDU_MANIFEST = OUT / 'urdu_asr_manifest.csv'urdu_count = 0if URDU_TSV.is_file() and URDU_AUDIO.is_dir():    audio_index = {p.stem: str(p) for p in URDU_AUDIO.rglob('*.wav') if p.is_file()}    speech = pd.read_csv(URDU_TSV, sep='\t', engine='python')    speech['audio_path'] = speech['path'].astype(str).map(lambda x: audio_index.get(Path(x).stem))    speech['text'] = speech['sentence'].fillna('').astype(str).str.strip()    speech = speech[speech['audio_path'].notna() & speech['text'].ne('') & speech['locale'].astype(str).eq('ur')].copy()    speech = speech.drop_duplicates(subset=['audio_path'])    speech[['audio_path','text','client_id']].to_csv(URDU_MANIFEST,index=False)    urdu_count = len(speech)print('Urdu ASR matched pairs:', urdu_count)CONFIG = { 'printed_english_ocr': {'run':False,'annotation_csv':'','image_col':'','text_col':''}, 'handwritten_english_ocr': {'run':False,'annotation_csv':'','image_col':'','text_col':''}, 'urdu_line_ocr': {'run':False,'csv_path':'','image_col':'image_path','text_col':'text'}, 'pkr_currency': {'run':pkr_dir is not None,'data_dir':str(pkr_dir) if pkr_dir else ''}, 'yolo_objects_person': {'run':True,'custom_data_yaml':'','test_image':''}, 'whisper_urdu_asr': {'run':urdu_count >= 100,'csv_path':str(URDU_MANIFEST) if urdu_count >= 100 else '', 'audio_col':'audio_path','text_col':'text','language':'urdu'}, 'whisper_english_asr': {'run':False,'csv_path':'','audio_col':'audio_path','text_col':'text','language':'english'},}MAX_NOTEBOOK_TIMEOUT=7200print('\\nONE-CLICK TRAINING PLAN')for name,cfg in CONFIG.items():    print(name, 'ENABLED' if cfg['run'] else 'SKIPPED — verified labels/config not available')

## 2. Fetch training code and validate the automatic selections
The source notebooks are downloaded automatically. You do not need Git, CMD, or manual path configuration.

In [ ]:
from pathlib import Pathimport json# Specialist notebooks are embedded here; no Git clone or private-repository login is needed.EMBEDDED_JSON = "{\"notebooks/currency/pkr_mobilenetv2_training_muhammad_farhan.ipynb\":{\"nbformat\":4,\"nbformat_minor\":5,\"metadata\":{\"kernelspec\":{\"display_name\":\"Python 3\",\"language\":\"python\",\"name\":\"python3\"},\"language_info\":{\"name\":\"python\",\"version\":\"3\"}},\"cells\":[{\"cell_type\":\"markdown\",\"metadata\":{},\"source\":[\"# NOORIX-AI PKR Currency Classification — MobileNetV2\\n\",\"**Prepared by Muhammad Farhan | NOORIX-AI**\\n\\n\",\"Classifies visible Pakistani banknote denomination/side labels only. It is NOT counterfeit detection or authentication.\\n\\n\",\"## Kaggle notebook title\\n`NOORIX-PKR-Currency-MobileNetV2-Muhammad-Farhan`\\n\\n\",\"## Dataset to attach\\nSearch Kaggle: `Pakistanicurrencydataset` (owner `muhammadammarabid`) or use the PKR dataset already downloaded. Dataset must be ImageFolder-style: one folder per class, images inside each class folder. Do not mix Afghani and PKR classes. Generic `Back`/`Front` folders are excluded.\\n\\n\",\"## Output\\n`NOORIX_PKR_MobileNetV2_Muhammad_Farhan.zip`\\n\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"!pip -q install -U onnx onnxscript scikit-learn\\n\",\"import os, glob, json, shutil, random, sys\\n\",\"from pathlib import Path\\n\",\"import numpy as np, torch, torch.nn as nn\\n\",\"from torch.utils.data import DataLoader, Subset\\n\",\"from torchvision import datasets, transforms, models\\n\",\"from sklearn.model_selection import train_test_split\\n\",\"from sklearn.metrics import classification_report, accuracy_score\\n\",\"SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)\\n\",\"if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)\\n\",\"device='cuda' if torch.cuda.is_available() else 'cpu'\\n\",\"print('Python:',sys.version); print('PyTorch:',torch.__version__); print('Device:',device)\\n\",\"if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))\\n\",\"ROOTS=[Path(p) for p in glob.glob('/kaggle/input/*')]\\n\",\"print('Kaggle inputs:'); [print(' -',p) for p in ROOTS]\\n\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Set this to the directory whose immediate subfolders are class names.\\n\",\"DATA_DIR=''\\n\",\"if not DATA_DIR: raise ValueError('Set DATA_DIR to the ImageFolder root shown under /kaggle/input.')\\n\",\"DATA_DIR=Path(DATA_DIR)\\n\",\"if not DATA_DIR.is_dir(): raise FileNotFoundError(f'DATA_DIR does not exist: {DATA_DIR}')\\n\",\"IMG_SIZE=224; BATCH_SIZE=16; EPOCHS=6\\n\",\"transform=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])\\n\",\"full=datasets.ImageFolder(str(DATA_DIR),transform=transform)\\n\",\"excluded={'back','front','reverse','obverse','background','unknown'}\\n\",\"labels=[name for name in full.classes if name.strip().lower() not in excluded]\\n\",\"if len(labels)<2: raise ValueError(f'Need at least 2 denomination classes after filtering. Found: {labels}')\\n\",\"old_to_new={full.class_to_idx[name]:i for i,name in enumerate(labels)}\\n\",\"indices=[i for i,target in enumerate(full.targets) if target in old_to_new]\\n\",\"targets=[old_to_new[full.targets[i]] for i in indices]\\n\",\"counts={name:sum(1 for y in targets if y==i) for i,name in enumerate(labels)}\\n\",\"print('Classes:',labels); print('Class counts:',counts); print('Usable images:',len(indices))\\n\",\"if len(indices)<20 or min(counts.values())<5: raise ValueError('Need at least 20 usable images and at least 5 images per class for a first training run.')\\n\",\"try: tr_idx,va_idx=train_test_split(np.arange(len(indices)),test_size=0.2,random_state=SEED,stratify=targets)\\n\",\"except ValueError as e: raise ValueError(f'Cannot create a stratified train/validation split: {e}')\\n\",\"class RemappedSubset(torch.utils.data.Dataset):\\n\",\"    def __init__(self,base,positions): self.base=base; self.positions=list(positions)\\n\",\"    def __len__(self): return len(self.positions)\\n\",\"    def __getitem__(self,idx):\\n\",\"        original_index=indices[self.positions[idx]]\\n\",\"        image,old_label=self.base[original_index]\\n\",\"        return image,old_to_new[old_label]\\n\",\"train_ds=RemappedSubset(full,tr_idx); val_ds=RemappedSubset(full,va_idx)\\n\",\"train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=(device=='cuda'))\\n\",\"val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=(device=='cuda'))\\n\",\"print('Train:',len(train_ds),'Validation:',len(val_ds))\\n\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"weights=models.MobileNet_V2_Weights.DEFAULT\\n\",\"model=models.mobilenet_v2(weights=weights)\\n\",\"model.classifier[1]=nn.Linear(model.last_channel,len(labels))\\n\",\"model=model.to(device)\\n\",\"criterion=nn.CrossEntropyLoss(); optimizer=torch.optim.AdamW(model.parameters(),lr=2e-4)\\n\",\"best_acc=-1.0; best_path=Path('/kaggle/working/mobilenetv2_pkr_best.pth')\\n\",\"for epoch in range(EPOCHS):\\n\",\"    model.train(); losses=[]\\n\",\"    for images,y in train_loader:\\n\",\"        images=images.to(device); y=y.to(device)\\n\",\"        optimizer.zero_grad(set_to_none=True); logits=model(images); loss=criterion(logits,y); loss.backward(); optimizer.step(); losses.append(loss.item())\\n\",\"    model.eval(); truth=[]; pred=[]\\n\",\"    with torch.no_grad():\\n\",\"        for images,y in val_loader:\\n\",\"            logits=model(images.to(device)); pred.extend(logits.argmax(1).cpu().tolist()); truth.extend(y.tolist())\\n\",\"    acc=accuracy_score(truth,pred)\\n\",\"    print(f'Epoch {epoch+1}/{EPOCHS} | train_loss={np.mean(losses):.4f} | val_accuracy={acc:.4f}')\\n\",\"    if acc>best_acc: best_acc=acc; torch.save(model.state_dict(),best_path)\\n\",\"if not best_path.exists(): raise RuntimeError('Training did not produce a checkpoint.')\\n\",\"print('Best validation accuracy:',best_acc)\\n\",\"print(classification_report(truth,pred,labels=list(range(len(labels))),target_names=labels,zero_division=0))\\n\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"model.load_state_dict(torch.load(best_path,map_location=device,weights_only=True)); model.eval()\\n\",\"out=Path('/kaggle/working/NOORIX_PKR_MobileNetV2_Muhammad_Farhan'); out.mkdir(parents=True,exist_ok=True)\\n\",\"torch.save(model.cpu().state_dict(),out/'mobilenetv2_pkr_best.pth')\\n\",\"(out/'labels.json').write_text(json.dumps({'classes':labels},ensure_ascii=False,indent=2),encoding='utf-8')\\n\",\"metadata={'project':'NOORIX-AI','prepared_by':'Muhammad Farhan','architecture':'MobileNetV2','input_size':[1,3,224,224],'normalization':{'mean':[0.485,0.456,0.406],'std':[0.229,0.224,0.225]},'validation_accuracy':float(best_acc),'train_samples':len(train_ds),'validation_samples':len(val_ds),'warning':'Denomination/side classification only; not counterfeit detection.'}\\n\",\"(out/'metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')\\n\",\"dummy=torch.zeros(1,3,224,224)\\n\",\"torch.onnx.export(model,dummy,str(out/'mobilenetv2_pkr.onnx'),input_names=['image'],output_names=['logits'],dynamic_axes={'image':{0:'batch'},'logits':{0:'batch'}},opset_version=17,dynamo=False)\\n\",\"zip_path=shutil.make_archive('/kaggle/working/NOORIX_PKR_MobileNetV2_Muhammad_Farhan','zip',out)\\n\",\"print('DOWNLOAD THIS FILE:',zip_path); print('ZIP contents:'); [print(' -',p.name) for p in out.iterdir()]\\n\"]}]},\"notebooks/detection/yolov8n_objects_person_muhammad_farhan.ipynb\":{\"cells\":[{\"cell_type\":\"markdown\",\"metadata\":{},\"source\":[\"# NOORIX-AI Everyday Object & Person Detection — YOLOv8n\\n\",\"**Prepared by Muhammad Farhan**\\n\",\"\\n\",\"This notebook uses the COCO-pretrained YOLOv8 nano model for everyday objects and people. COCO already includes the `person` class, so a separate person model is not normally needed for basic person detection. Optional custom training is supported if you have a labelled YOLO dataset.\\n\",\"\\n\",\"## Kaggle notebook title\\n\",\"`NOORIX-Object-Person-YOLOv8n-Muhammad-Farhan`\\n\",\"\\n\",\"## Dataset to add\\n\",\"- For pretrained COCO inference/export: **no dataset input is required**; model weights are fetched by Ultralytics when internet access is enabled.\\n\",\"- For custom training: add your own YOLO-format dataset with `data.yaml`, `images/train`, `images/val`, `labels/train`, `labels/val`. Kaggle search term: `COCO 2017 object detection YOLO format`. Ultralytics COCO documentation describes the 80 everyday classes, including person.\\n\",\"\\n\",\"Do not assume this YOLO ONNX output plugs directly into NOORIX's current SSD adapter. A YOLO-specific decoder/adapter and numerical validation are required before app integration.\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"!pip -q install ultralytics onnx\\n\",\"from pathlib import Path\\n\",\"import json, shutil, torch\\n\",\"from ultralytics import YOLO\\n\",\"print(\\\"GPU:\\\",torch.cuda.get_device_name(0) if torch.cuda.is_available() else \\\"CPU\\\")\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"# First run pretrained COCO model. YOLOv8n is the lightweight option for CPU deployment experiments.\\n\",\"model=YOLO(\\\"yolov8n.pt\\\")\\n\",\"print(\\\"COCO classes:\\\",len(model.names))\\n\",\"print(\\\"Person class id:\\\",[k for k,v in model.names.items() if v==\\\"person\\\"])\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"# Optional custom training. Leave CUSTOM_DATA_YAML empty to skip training and use COCO pretrained weights.\\n\",\"CUSTOM_DATA_YAML=\\\"\\\" # e.g. \\\"/kaggle/input/my-yolo-data/data.yaml\\\"\\n\",\"if CUSTOM_DATA_YAML:\\n\",\"    model.train(data=CUSTOM_DATA_YAML,epochs=30,imgsz=640,batch=16,device=0 if torch.cuda.is_available() else \\\"cpu\\\",\\n\",\"                project=\\\"/kaggle/working/noorix_yolo\\\",name=\\\"custom\\\",patience=8)\\n\",\"    model=YOLO(\\\"/kaggle/working/noorix_yolo/custom/weights/best.pt\\\")\\n\",\"else:\\n\",\"    print(\\\"Skipping custom training; using pretrained COCO model.\\\")\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"# Evaluate on a sample image or dataset. Set TEST_IMAGE to a file from an attached image dataset.\\n\",\"TEST_IMAGE=\\\"\\\"\\n\",\"if TEST_IMAGE:\\n\",\"    results=model.predict(TEST_IMAGE,conf=0.35,save=True)\\n\",\"    for r in results:\\n\",\"        for b in r.boxes:\\n\",\"            print(model.names[int(b.cls[0])],float(b.conf[0]),b.xyxy[0].tolist())\\n\",\"else:\\n\",\"    print(\\\"Set TEST_IMAGE to inspect a sample, or attach a labelled dataset and use model.val(data=...).\\\")\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"# Export ONNX and package the .pt + ONNX + class metadata.\\n\",\"exported=model.export(format=\\\"onnx\\\",imgsz=640,dynamic=False)\\n\",\"out=Path(\\\"/kaggle/working/NOORIX_Object_Person_YOLOv8n_Muhammad_Farhan\\\"); out.mkdir(exist_ok=True)\\n\",\"shutil.copy2(model.ckpt_path,out/\\\"best.pt\\\") if getattr(model,\\\"ckpt_path\\\",None) and Path(model.ckpt_path).exists() else shutil.copy2(\\\"yolov8n.pt\\\",out/\\\"yolov8n.pt\\\")\\n\",\"shutil.copy2(exported,out/Path(exported).name)\\n\",\"(out/\\\"labels.json\\\").write_text(json.dumps({str(k):v for k,v in model.names.items()},indent=2),encoding=\\\"utf-8\\\")\\n\",\"(out/\\\"metadata.json\\\").write_text(json.dumps({\\\"project\\\":\\\"NOORIX-AI\\\",\\\"prepared_by\\\":\\\"Muhammad Farhan\\\",\\\"model\\\":\\\"YOLOv8n\\\",\\\"classes\\\":len(model.names),\\n\",\"\\\"input_size\\\":[1,3,640,640],\\\"note\\\":\\\"YOLO ONNX needs its own decoder; not compatible with the current SSD output adapter.\\\"},indent=2),encoding=\\\"utf-8\\\")\\n\",\"z=shutil.make_archive(\\\"/kaggle/working/NOORIX_Object_Person_YOLOv8n_Muhammad_Farhan\\\",\\\"zip\\\",out)\\n\",\"print(\\\"ZIP:\\\",z)\\n\",\"\"]}],\"metadata\":{\"kernelspec\":{\"display_name\":\"Python 3\",\"language\":\"python\",\"name\":\"python3\"},\"language_info\":{\"name\":\"python\",\"version\":\"3\"}},\"nbformat\":4,\"nbformat_minor\":5},\"notebooks/ocr/printed_ocr_training_muhammad_farhan.ipynb\":{\"cells\":[{\"cell_type\":\"markdown\",\"metadata\":{},\"source\":[\"# NOORIX-AI Printed OCR Training\\n\",\"**Prepared by Muhammad Farhan | NOORIX-AI**\\n\",\"\\\\n\",\"This notebook fine-tunes a pretrained TrOCR model on your own labelled printed image dataset. It evaluates Character Error Rate (CER), Word Error Rate (WER), and saves the trained model. It does not magically create labels: each training image must have its exact transcription.\\n\",\"\\\\n\",\"**Language note:** The selected pretrained checkpoint is English-oriented. Urdu can be included only as an experimental fine-tuning task when your CSV contains accurate Urdu transcriptions and enough representative labelled data. Do not treat an English-only checkpoint as validated Urdu OCR.\\n\",\"\\\\n\",\"## Dataset CSV format\\n\",\"Create a CSV with two columns: `image_path` and `text`. Each row is one image containing a line (or short text crop) and its exact transcription. Example:\\n\",\"\\\\n\",\"| image_path | text |\\n\",\"|---|---|\\n\",\"| lines/img_0001.png | This is a printed line. |\\n\",\"| lines/img_0002.png | Exact text from the second line. |\\n\",\"\\\\n\",\"Paths may be absolute Kaggle paths or relative to the dataset folder. Split train/validation automatically if no split column is provided.\\n\",\"\\\\n\",\"### Use printed book/document line crops. For Urdu, use accurately transcribed Urdu rows; the English pretrained model is only a starting point and must be evaluated carefully.\\n\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Kaggle OCR training notebook\\n\",\"import os, glob, json, random, zipfile, shutil, textwrap\\n\",\"from pathlib import Path\\n\",\"import pandas as pd\\n\",\"import numpy as np\\n\",\"from PIL import Image\\n\",\"import torch\\n\",\"print(\\\"Python:\\\", __import__(\\\"sys\\\").version)\\n\",\"print(\\\"PyTorch:\\\", torch.__version__)\\n\",\"print(\\\"GPU:\\\", torch.cuda.get_device_name(0) if torch.cuda.is_available() else \\\"CPU\\\")\\n\",\"INPUT_ROOTS = [Path(p) for p in glob.glob(\\\"/kaggle/input/*\\\")]\\n\",\"print(\\\"Kaggle inputs:\\\")\\n\",\"for p in INPUT_ROOTS: print(\\\" -\\\", p)\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Find annotation CSV files. A usable CSV needs one image-path column and one transcription column.\\n\",\"CSV_CANDIDATES = []\\n\",\"for root in INPUT_ROOTS:\\n\",\"    CSV_CANDIDATES += list(root.rglob(\\\"*.csv\\\"))\\n\",\"print(\\\"CSV files found:\\\", len(CSV_CANDIDATES))\\n\",\"for i, p in enumerate(CSV_CANDIDATES[:80]):\\n\",\"    print(f\\\"[{i}] {p}\\\")\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Select your annotation CSV after reviewing the list above.\\n\",\"# Required columns: image_path (or file_name/path/image) and text (or transcription/label).\\n\",\"# If your dataset has no transcription CSV, create one and upload it as a Kaggle Dataset.\\n\",\"ANNOTATION_CSV = \\\"\\\"  # e.g. \\\"/kaggle/input/my-ocr-data/metadata.csv\\\"\\n\",\"IMAGE_COL = \\\"\\\"       # leave blank for auto-detection\\n\",\"TEXT_COL = \\\"\\\"        # leave blank for auto-detection\\n\",\"\\n\",\"if not ANNOTATION_CSV:\\n\",\"    raise ValueError(\\\"Set ANNOTATION_CSV to the CSV containing image paths and exact transcriptions. Folder names alone are not OCR ground-truth text.\\\")\\n\",\"df = pd.read_csv(ANNOTATION_CSV)\\n\",\"print(\\\"Columns:\\\", list(df.columns))\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Normalize annotation columns and resolve image paths.\\n\",\"from pathlib import Path\\n\",\"import os\\n\",\"cols_lower = {c.lower().strip(): c for c in df.columns}\\n\",\"def choose_col(explicit, options):\\n\",\"    if explicit:\\n\",\"        if explicit not in df.columns: raise ValueError(f\\\"Column {explicit!r} not found\\\")\\n\",\"        return explicit\\n\",\"    for name in options:\\n\",\"        if name in cols_lower: return cols_lower[name]\\n\",\"    raise ValueError(f\\\"Could not find any of these columns: {options}. Set IMAGE_COL/TEXT_COL manually.\\\")\\n\",\"img_col = choose_col(IMAGE_COL, [\\\"image_path\\\",\\\"filepath\\\",\\\"file_path\\\",\\\"file_name\\\",\\\"filename\\\",\\\"path\\\",\\\"image\\\"])\\n\",\"txt_col = choose_col(TEXT_COL, [\\\"text\\\",\\\"transcription\\\",\\\"transcript\\\",\\\"label\\\",\\\"ground_truth\\\",\\\"gt\\\"])\\n\",\"def resolve_path(value):\\n\",\"    p = Path(str(value))\\n\",\"    if p.is_file(): return str(p)\\n\",\"    for root in INPUT_ROOTS:\\n\",\"        q = root / p\\n\",\"        if q.is_file(): return str(q)\\n\",\"        matches = list(root.rglob(p.name))\\n\",\"        if matches: return str(matches[0])\\n\",\"    return None\\n\",\"df = df[[img_col, txt_col]].copy()\\n\",\"df.columns = [\\\"image_path\\\", \\\"text\\\"]\\n\",\"df[\\\"image_path\\\"] = df[\\\"image_path\\\"].apply(resolve_path)\\n\",\"df[\\\"text\\\"] = df[\\\"text\\\"].fillna(\\\"\\\").astype(str).str.strip()\\n\",\"df = df.dropna(subset=[\\\"image_path\\\"])\\n\",\"df = df[df[\\\"text\\\"].str.len() > 0].reset_index(drop=True)\\n\",\"print(\\\"Usable labelled rows:\\\", len(df))\\n\",\"display(df.head())\\n\",\"if len(df) < 20: raise ValueError(\\\"Too few labelled rows. Add a larger correctly transcribed dataset.\\\")\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Train/validation split and basic image integrity checks.\\n\",\"from sklearn.model_selection import train_test_split\\n\",\"random.seed(42)\\n\",\"train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)\\n\",\"print(\\\"Train:\\\", len(train_df), \\\"Validation:\\\", len(val_df))\\n\",\"def check_image(path):\\n\",\"    try:\\n\",\"        with Image.open(path) as im: im.verify()\\n\",\"        return True\\n\",\"    except Exception: return False\\n\",\"for part in (train_df, val_df):\\n\",\"    part[\\\"valid\\\"] = part[\\\"image_path\\\"].apply(check_image)\\n\",\"    part.drop(part[~part[\\\"valid\\\"]].index, inplace=True)\\n\",\"print(\\\"After image checks:\\\", len(train_df), len(val_df))\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"!pip -q install -U transformers accelerate jiwer sentencepiece evaluate\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments\\n\",\"from torch.utils.data import Dataset\\n\",\"import evaluate\\n\",\"MODEL_NAME = \\\"microsoft/trocr-base-printed\\\"\\n\",\"processor = TrOCRProcessor.from_pretrained(MODEL_NAME)\\n\",\"model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)\\n\",\"model.config.decoder_start_token_id = processor.tokenizer.cls_token_id\\n\",\"model.config.pad_token_id = processor.tokenizer.pad_token_id\\n\",\"model.config.eos_token_id = processor.tokenizer.sep_token_id\\n\",\"model.config.vocab_size = model.config.decoder.vocab_size\\n\",\"model.config.max_length = 128\\n\",\"model.config.early_stopping = True\\n\",\"model.config.no_repeat_ngram_size = 3\\n\",\"model.config.length_penalty = 2.0\\n\",\"if torch.cuda.is_available():\\n\",\"    model.to(\\\"cuda\\\")\\n\",\"print(\\\"Loaded:\\\", MODEL_NAME)\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"class OCRDataset(Dataset):\\n\",\"    def __init__(self, frame, processor, max_target_length=128):\\n\",\"        self.frame = frame.reset_index(drop=True)\\n\",\"        self.processor = processor\\n\",\"        self.max_target_length = max_target_length\\n\",\"    def __len__(self): return len(self.frame)\\n\",\"    def __getitem__(self, idx):\\n\",\"        row = self.frame.iloc[idx]\\n\",\"        image = Image.open(row.image_path).convert(\\\"RGB\\\")\\n\",\"        pixel_values = self.processor(images=image, return_tensors=\\\"pt\\\").pixel_values.squeeze(0)\\n\",\"        labels = self.processor.tokenizer(\\n\",\"            row.text, padding=\\\"max_length\\\", max_length=self.max_target_length,\\n\",\"            truncation=True, return_tensors=\\\"pt\\\"\\n\",\"        ).input_ids.squeeze(0)\\n\",\"        labels[labels == self.processor.tokenizer.pad_token_id] = -100\\n\",\"        return {\\\"pixel_values\\\": pixel_values, \\\"labels\\\": labels}\\n\",\"train_ds = OCRDataset(train_df, processor)\\n\",\"val_ds = OCRDataset(val_df, processor)\\n\",\"def collate(batch):\\n\",\"    return {\\n\",\"        \\\"pixel_values\\\": torch.stack([x[\\\"pixel_values\\\"] for x in batch]),\\n\",\"        \\\"labels\\\": torch.stack([x[\\\"labels\\\"] for x in batch])\\n\",\"    }\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Fine-tuning. Start with 3 epochs; increase only after checking validation CER.\\n\",\"cer_metric = evaluate.load(\\\"cer\\\")\\n\",\"wer_metric = evaluate.load(\\\"wer\\\")\\n\",\"def compute_metrics(pred):\\n\",\"    pred_ids = pred.predictions\\n\",\"    if isinstance(pred_ids, tuple): pred_ids = pred_ids[0]\\n\",\"    label_ids = pred.label_ids.copy()\\n\",\"    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id\\n\",\"    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)\\n\",\"    label_text = processor.batch_decode(label_ids, skip_special_tokens=True)\\n\",\"    return {\\n\",\"        \\\"cer\\\": cer_metric.compute(predictions=pred_text, references=label_text),\\n\",\"        \\\"wer\\\": wer_metric.compute(predictions=pred_text, references=label_text)\\n\",\"    }\\n\",\"args = Seq2SeqTrainingArguments(\\n\",\"    output_dir=\\\"/kaggle/working/noorix_ocr_checkpoints\\\",\\n\",\"    predict_with_generate=True,\\n\",\"    num_train_epochs=3,\\n\",\"    per_device_train_batch_size=4,\\n\",\"    per_device_eval_batch_size=4,\\n\",\"    learning_rate=5e-5,\\n\",\"    fp16=torch.cuda.is_available(),\\n\",\"    eval_strategy=\\\"epoch\\\",\\n\",\"    save_strategy=\\\"epoch\\\",\\n\",\"    logging_steps=50,\\n\",\"    save_total_limit=2,\\n\",\"    load_best_model_at_end=True,\\n\",\"    metric_for_best_model=\\\"cer\\\",\\n\",\"    greater_is_better=False,\\n\",\"    report_to=\\\"none\\\"\\n\",\")\\n\",\"trainer = Seq2SeqTrainer(\\n\",\"    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,\\n\",\"    data_collator=collate, compute_metrics=compute_metrics\\n\",\")\\n\",\"trainer.train()\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Final validation metrics and a few sample predictions.\\n\",\"metrics = trainer.evaluate()\\n\",\"print(metrics)\\n\",\"device = \\\"cuda\\\" if torch.cuda.is_available() else \\\"cpu\\\"\\n\",\"model.to(device)\\n\",\"def predict(path):\\n\",\"    image = Image.open(path).convert(\\\"RGB\\\")\\n\",\"    px = processor(images=image, return_tensors=\\\"pt\\\").pixel_values.to(device)\\n\",\"    with torch.no_grad():\\n\",\"        ids = model.generate(px, max_new_tokens=128)\\n\",\"    return processor.batch_decode(ids, skip_special_tokens=True)[0]\\n\",\"for _, row in val_df.head(8).iterrows():\\n\",\"    print(\\\"\\\\nREFERENCE:\\\", row.text)\\n\",\"    print(\\\"PREDICTED:\\\", predict(row.image_path))\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Save model, processor, metadata and one ZIP for download.\\n\",\"OUT = Path(\\\"/kaggle/working/NOORIX_PRINTED_OCR_Muhammad_Farhan\\\")\\n\",\"OUT.mkdir(parents=True, exist_ok=True)\\n\",\"model.save_pretrained(OUT / \\\"model\\\")\\n\",\"processor.save_pretrained(OUT / \\\"processor\\\")\\n\",\"meta = {\\n\",\"    \\\"project\\\": \\\"NOORIX-AI\\\", \\\"prepared_by\\\": \\\"Muhammad Farhan\\\",\\n\",\"    \\\"task\\\": \\\"printed OCR\\\", \\\"base_model\\\": MODEL_NAME,\\n\",\"    \\\"train_samples\\\": len(train_df), \\\"validation_samples\\\": len(val_df),\\n\",\"    \\\"validation_metrics\\\": {k: float(v) for k,v in metrics.items() if isinstance(v, (int,float))},\\n\",\"    \\\"languages\\\": \\\"Checkpoint is English-oriented; Urdu fine-tuning is experimental and requires verified Urdu labels.\\\",\\n\",\"    \\\"warning\\\": \\\"Evaluate on a separate real-world test set before integration.\\\"\\n\",\"}\\n\",\"(OUT / \\\"metadata.json\\\").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding=\\\"utf-8\\\")\\n\",\"zip_path = shutil.make_archive(str(OUT), \\\"zip\\\", OUT)\\n\",\"print(\\\"Saved:\\\", zip_path)\\n\",\"\\n\",\"\"]},{\"cell_type\":\"markdown\",\"metadata\":{},\"source\":[\"## Before integrating into NOORIX-AI\\n\",\"Check validation CER/WER and visually inspect predictions. Lower CER/WER is better. Keep the original held-out test set untouched. Do not claim Urdu support or production accuracy until tested on a representative Urdu test set. The saved Hugging Face model folder is the primary artifact; ONNX export should be handled in a separate compatibility-tested step.\\n\"]}],\"metadata\":{\"kernelspec\":{\"display_name\":\"Python 3\",\"language\":\"python\",\"name\":\"python3\"},\"language_info\":{\"name\":\"python\",\"version\":\"3\"}},\"nbformat\":4,\"nbformat_minor\":5},\"notebooks/ocr/handwritten_ocr_training_muhammad_farhan.ipynb\":{\"cells\":[{\"cell_type\":\"markdown\",\"metadata\":{},\"source\":[\"# NOORIX-AI Handwritten OCR Training\\n\",\"**Prepared by Muhammad Farhan | NOORIX-AI**\\n\",\"\\\\n\",\"This notebook fine-tunes a pretrained TrOCR model on your own labelled handwritten image dataset. It evaluates Character Error Rate (CER), Word Error Rate (WER), and saves the trained model. It does not magically create labels: each training image must have its exact transcription.\\n\",\"\\\\n\",\"**Language note:** The selected pretrained checkpoint is English-oriented. Urdu can be included only as an experimental fine-tuning task when your CSV contains accurate Urdu transcriptions and enough representative labelled data. Do not treat an English-only checkpoint as validated Urdu OCR.\\n\",\"\\\\n\",\"## Dataset CSV format\\n\",\"Create a CSV with two columns: `image_path` and `text`. Each row is one image containing a line (or short text crop) and its exact transcription. Example:\\n\",\"\\\\n\",\"| image_path | text |\\n\",\"|---|---|\\n\",\"| lines/img_0001.png | This is a printed line. |\\n\",\"| lines/img_0002.png | Exact text from the second line. |\\n\",\"\\\\n\",\"Paths may be absolute Kaggle paths or relative to the dataset folder. Split train/validation automatically if no split column is provided.\\n\",\"\\\\n\",\"### Use handwritten line crops with exact transcriptions. Handwriting varies heavily; use a representative dataset and inspect errors before claiming quality.\\n\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Kaggle OCR training notebook\\n\",\"import os, glob, json, random, zipfile, shutil, textwrap\\n\",\"from pathlib import Path\\n\",\"import pandas as pd\\n\",\"import numpy as np\\n\",\"from PIL import Image\\n\",\"import torch\\n\",\"print(\\\"Python:\\\", __import__(\\\"sys\\\").version)\\n\",\"print(\\\"PyTorch:\\\", torch.__version__)\\n\",\"print(\\\"GPU:\\\", torch.cuda.get_device_name(0) if torch.cuda.is_available() else \\\"CPU\\\")\\n\",\"INPUT_ROOTS = [Path(p) for p in glob.glob(\\\"/kaggle/input/*\\\")]\\n\",\"print(\\\"Kaggle inputs:\\\")\\n\",\"for p in INPUT_ROOTS: print(\\\" -\\\", p)\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Find annotation CSV files. A usable CSV needs one image-path column and one transcription column.\\n\",\"CSV_CANDIDATES = []\\n\",\"for root in INPUT_ROOTS:\\n\",\"    CSV_CANDIDATES += list(root.rglob(\\\"*.csv\\\"))\\n\",\"print(\\\"CSV files found:\\\", len(CSV_CANDIDATES))\\n\",\"for i, p in enumerate(CSV_CANDIDATES[:80]):\\n\",\"    print(f\\\"[{i}] {p}\\\")\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Select your annotation CSV after reviewing the list above.\\n\",\"# Required columns: image_path (or file_name/path/image) and text (or transcription/label).\\n\",\"# If your dataset has no transcription CSV, create one and upload it as a Kaggle Dataset.\\n\",\"ANNOTATION_CSV = \\\"\\\"  # e.g. \\\"/kaggle/input/my-ocr-data/metadata.csv\\\"\\n\",\"IMAGE_COL = \\\"\\\"       # leave blank for auto-detection\\n\",\"TEXT_COL = \\\"\\\"        # leave blank for auto-detection\\n\",\"\\n\",\"if not ANNOTATION_CSV:\\n\",\"    raise ValueError(\\\"Set ANNOTATION_CSV to the CSV containing image paths and exact transcriptions. Folder names alone are not OCR ground-truth text.\\\")\\n\",\"df = pd.read_csv(ANNOTATION_CSV)\\n\",\"print(\\\"Columns:\\\", list(df.columns))\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Normalize annotation columns and resolve image paths.\\n\",\"from pathlib import Path\\n\",\"import os\\n\",\"cols_lower = {c.lower().strip(): c for c in df.columns}\\n\",\"def choose_col(explicit, options):\\n\",\"    if explicit:\\n\",\"        if explicit not in df.columns: raise ValueError(f\\\"Column {explicit!r} not found\\\")\\n\",\"        return explicit\\n\",\"    for name in options:\\n\",\"        if name in cols_lower: return cols_lower[name]\\n\",\"    raise ValueError(f\\\"Could not find any of these columns: {options}. Set IMAGE_COL/TEXT_COL manually.\\\")\\n\",\"img_col = choose_col(IMAGE_COL, [\\\"image_path\\\",\\\"filepath\\\",\\\"file_path\\\",\\\"file_name\\\",\\\"filename\\\",\\\"path\\\",\\\"image\\\"])\\n\",\"txt_col = choose_col(TEXT_COL, [\\\"text\\\",\\\"transcription\\\",\\\"transcript\\\",\\\"label\\\",\\\"ground_truth\\\",\\\"gt\\\"])\\n\",\"def resolve_path(value):\\n\",\"    p = Path(str(value))\\n\",\"    if p.is_file(): return str(p)\\n\",\"    for root in INPUT_ROOTS:\\n\",\"        q = root / p\\n\",\"        if q.is_file(): return str(q)\\n\",\"        matches = list(root.rglob(p.name))\\n\",\"        if matches: return str(matches[0])\\n\",\"    return None\\n\",\"df = df[[img_col, txt_col]].copy()\\n\",\"df.columns = [\\\"image_path\\\", \\\"text\\\"]\\n\",\"df[\\\"image_path\\\"] = df[\\\"image_path\\\"].apply(resolve_path)\\n\",\"df[\\\"text\\\"] = df[\\\"text\\\"].fillna(\\\"\\\").astype(str).str.strip()\\n\",\"df = df.dropna(subset=[\\\"image_path\\\"])\\n\",\"df = df[df[\\\"text\\\"].str.len() > 0].reset_index(drop=True)\\n\",\"print(\\\"Usable labelled rows:\\\", len(df))\\n\",\"display(df.head())\\n\",\"if len(df) < 20: raise ValueError(\\\"Too few labelled rows. Add a larger correctly transcribed dataset.\\\")\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Train/validation split and basic image integrity checks.\\n\",\"from sklearn.model_selection import train_test_split\\n\",\"random.seed(42)\\n\",\"train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)\\n\",\"print(\\\"Train:\\\", len(train_df), \\\"Validation:\\\", len(val_df))\\n\",\"def check_image(path):\\n\",\"    try:\\n\",\"        with Image.open(path) as im: im.verify()\\n\",\"        return True\\n\",\"    except Exception: return False\\n\",\"for part in (train_df, val_df):\\n\",\"    part[\\\"valid\\\"] = part[\\\"image_path\\\"].apply(check_image)\\n\",\"    part.drop(part[~part[\\\"valid\\\"]].index, inplace=True)\\n\",\"print(\\\"After image checks:\\\", len(train_df), len(val_df))\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"!pip -q install -U transformers accelerate jiwer sentencepiece evaluate\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments\\n\",\"from torch.utils.data import Dataset\\n\",\"import evaluate\\n\",\"MODEL_NAME = \\\"microsoft/trocr-base-handwritten\\\"\\n\",\"processor = TrOCRProcessor.from_pretrained(MODEL_NAME)\\n\",\"model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)\\n\",\"model.config.decoder_start_token_id = processor.tokenizer.cls_token_id\\n\",\"model.config.pad_token_id = processor.tokenizer.pad_token_id\\n\",\"model.config.eos_token_id = processor.tokenizer.sep_token_id\\n\",\"model.config.vocab_size = model.config.decoder.vocab_size\\n\",\"model.config.max_length = 128\\n\",\"model.config.early_stopping = True\\n\",\"model.config.no_repeat_ngram_size = 3\\n\",\"model.config.length_penalty = 2.0\\n\",\"if torch.cuda.is_available():\\n\",\"    model.to(\\\"cuda\\\")\\n\",\"print(\\\"Loaded:\\\", MODEL_NAME)\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"class OCRDataset(Dataset):\\n\",\"    def __init__(self, frame, processor, max_target_length=128):\\n\",\"        self.frame = frame.reset_index(drop=True)\\n\",\"        self.processor = processor\\n\",\"        self.max_target_length = max_target_length\\n\",\"    def __len__(self): return len(self.frame)\\n\",\"    def __getitem__(self, idx):\\n\",\"        row = self.frame.iloc[idx]\\n\",\"        image = Image.open(row.image_path).convert(\\\"RGB\\\")\\n\",\"        pixel_values = self.processor(images=image, return_tensors=\\\"pt\\\").pixel_values.squeeze(0)\\n\",\"        labels = self.processor.tokenizer(\\n\",\"            row.text, padding=\\\"max_length\\\", max_length=self.max_target_length,\\n\",\"            truncation=True, return_tensors=\\\"pt\\\"\\n\",\"        ).input_ids.squeeze(0)\\n\",\"        labels[labels == self.processor.tokenizer.pad_token_id] = -100\\n\",\"        return {\\\"pixel_values\\\": pixel_values, \\\"labels\\\": labels}\\n\",\"train_ds = OCRDataset(train_df, processor)\\n\",\"val_ds = OCRDataset(val_df, processor)\\n\",\"def collate(batch):\\n\",\"    return {\\n\",\"        \\\"pixel_values\\\": torch.stack([x[\\\"pixel_values\\\"] for x in batch]),\\n\",\"        \\\"labels\\\": torch.stack([x[\\\"labels\\\"] for x in batch])\\n\",\"    }\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Fine-tuning. Start with 3 epochs; increase only after checking validation CER.\\n\",\"cer_metric = evaluate.load(\\\"cer\\\")\\n\",\"wer_metric = evaluate.load(\\\"wer\\\")\\n\",\"def compute_metrics(pred):\\n\",\"    pred_ids = pred.predictions\\n\",\"    if isinstance(pred_ids, tuple): pred_ids = pred_ids[0]\\n\",\"    label_ids = pred.label_ids.copy()\\n\",\"    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id\\n\",\"    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)\\n\",\"    label_text = processor.batch_decode(label_ids, skip_special_tokens=True)\\n\",\"    return {\\n\",\"        \\\"cer\\\": cer_metric.compute(predictions=pred_text, references=label_text),\\n\",\"        \\\"wer\\\": wer_metric.compute(predictions=pred_text, references=label_text)\\n\",\"    }\\n\",\"args = Seq2SeqTrainingArguments(\\n\",\"    output_dir=\\\"/kaggle/working/noorix_ocr_checkpoints\\\",\\n\",\"    predict_with_generate=True,\\n\",\"    num_train_epochs=3,\\n\",\"    per_device_train_batch_size=4,\\n\",\"    per_device_eval_batch_size=4,\\n\",\"    learning_rate=5e-5,\\n\",\"    fp16=torch.cuda.is_available(),\\n\",\"    eval_strategy=\\\"epoch\\\",\\n\",\"    save_strategy=\\\"epoch\\\",\\n\",\"    logging_steps=50,\\n\",\"    save_total_limit=2,\\n\",\"    load_best_model_at_end=True,\\n\",\"    metric_for_best_model=\\\"cer\\\",\\n\",\"    greater_is_better=False,\\n\",\"    report_to=\\\"none\\\"\\n\",\")\\n\",\"trainer = Seq2SeqTrainer(\\n\",\"    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,\\n\",\"    data_collator=collate, compute_metrics=compute_metrics\\n\",\")\\n\",\"trainer.train()\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Final validation metrics and a few sample predictions.\\n\",\"metrics = trainer.evaluate()\\n\",\"print(metrics)\\n\",\"device = \\\"cuda\\\" if torch.cuda.is_available() else \\\"cpu\\\"\\n\",\"model.to(device)\\n\",\"def predict(path):\\n\",\"    image = Image.open(path).convert(\\\"RGB\\\")\\n\",\"    px = processor(images=image, return_tensors=\\\"pt\\\").pixel_values.to(device)\\n\",\"    with torch.no_grad():\\n\",\"        ids = model.generate(px, max_new_tokens=128)\\n\",\"    return processor.batch_decode(ids, skip_special_tokens=True)[0]\\n\",\"for _, row in val_df.head(8).iterrows():\\n\",\"    print(\\\"\\\\nREFERENCE:\\\", row.text)\\n\",\"    print(\\\"PREDICTED:\\\", predict(row.image_path))\\n\",\"\\n\",\"\"]},{\"cell_type\":\"code\",\"execution_count\":null,\"metadata\":{},\"outputs\":[],\"source\":[\"# Save model, processor, metadata and one ZIP for download.\\n\",\"OUT = Path(\\\"/kaggle/working/NOORIX_HANDWRITTEN_OCR_Muhammad_Farhan\\\")\\n\",\"OUT.mkdir(parents=True, exist_ok=True)\\n\",\"model.save_pretrained(OUT / \\\"model\\\")\\n\",\"processor.save_pretrained(OUT / \\\"processor\\\")\\n\",\"meta = {\\n\",\"    \\\"project\\\": \\\"NOORIX-AI\\\", \\\"prepared_by\\\": \\\"Muhammad Farhan\\\",\\n\",\"    \\\"task\\\": \\\"handwritten OCR\\\", \\\"base_model\\\": MODEL_NAME,\\n\",\"    \\\"train_samples\\\": len(train_df), \\\"validation_samples\\\": len(val_df),\\n\",\"    \\\"validation_metrics\\\": {k: float(v) for k,v in metrics.items() if isinstance(v, (int,float))},\\n\",\"    \\\"languages\\\": \\\"Checkpoint is English-oriented; Urdu fine-tuning is experimental and requires verified Urdu labels.\\\",\\n\",\"    \\\"warning\\\": \\\"Evaluate on a separate real-world test set before integration.\\\"\\n\",\"}\\n\",\"(OUT / \\\"metadata.json\\\").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding=\\\"utf-8\\\")\\n\",\"zip_path = shutil.make_archive(str(OUT), \\\"zip\\\", OUT)\\n\",\"print(\\\"Saved:\\\", zip_path)\\n\",\"\\n\",\"\"]},{\"cell_type\":\"markdown\",\"metadata\":{},\"source\":[\"## Before integrating into NOORIX-AI\\n\",\"Check validation CER/WER and visually inspect predictions. Lower CER/WER is better. Keep the original held-out test set untouched. Do not claim Urdu support or production accuracy until tested on a representative Urdu test set. The saved Hugging Face model folder is the primary artifact; ONNX export should be handled in a separate compatibility-tested step.\\n\"]}],\"metadata\":{\"kernelspec\":{\"display_name\":\"Python 3\",\"language\":\"python\",\"name\":\"python3\"},\"language_info\":{\"name\":\"python\",\"version\":\"3\"}},\"nbformat\":4,\"nbformat_minor\":5},\"notebooks/ocr/urdu_printed_handwritten_crnn_ctc_muhammad_farhan.ipynb\":{\"cells\":[{\"cell_type\":\"markdown\",\"metadata\":{},\"source\":[\"# NOORIX-AI Urdu Printed + Handwritten Line OCR — CRNN/CTC\\n\",\"**Prepared by Muhammad Farhan**\\n\",\"\\n\",\"This is an Urdu-specific line-recognition starter. Run it separately for printed Urdu or handwritten Urdu by attaching one dataset at a time. It learns Unicode Urdu characters from your labelled line images and evaluates Character Error Rate (CER). It is not a page-layout/text-detection model; input images must already be cropped to individual text lines.\\n\",\"\\n\",\"## Dataset to add\\n\",\"- **Printed Urdu:** search/add `UPTI 1.0 Urdu Printed Text Image Database` or use the **IIIT-Urdu OCR Dataset** (line images and transcriptions).\\n\",\"- **Handwritten Urdu:** use **NUST-UHWR** (line images with Unicode ground-truth transcriptions; official split files are available).\\n\",\"- Kaggle dataset search terms: `Urdu OCR line images transcription`, `Urdu handwriting recognition dataset`, `NUST UHWR Urdu handwriting`.\\n\",\"\\n\",\"## Kaggle notebook title\\n\",\"`NOORIX-Urdu-Printed-Handwritten-CRNN-CTC-Muhammad-Farhan`\\n\",\"\\n\",\"## Required manifest\\n\",\"Create a UTF-8 CSV with columns `image_path,text`, one line-image per row. For NUST-UHWR, convert the official tab-separated train/val/test manifest to CSV while preserving Urdu text. Do not train from page images without line crops and exact transcriptions. Keep an independent test set aside.\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"!pip -q install jiwer\\n\",\"import os, json, glob, math, shutil, random\\n\",\"from pathlib import Path\\n\",\"import numpy as np, pandas as pd\\n\",\"from PIL import Image, ImageOps\\n\",\"import torch, torch.nn as nn\\n\",\"from torch.utils.data import Dataset, DataLoader\\n\",\"print(\\\"GPU:\\\",torch.cuda.get_device_name(0) if torch.cuda.is_available() else \\\"CPU\\\")\\n\",\"for p in glob.glob(\\\"/kaggle/input/*\\\"): print(p)\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"# Select a CSV manifest for ONE dataset and ONE script type per run.\\n\",\"CSV_PATH=\\\"\\\" # e.g. \\\"/kaggle/input/nust-uhwr/metadata.csv\\\"\\n\",\"IMAGE_COL=\\\"image_path\\\"; TEXT_COL=\\\"text\\\"\\n\",\"if not CSV_PATH: raise ValueError(\\\"Set CSV_PATH to your UTF-8 line-image transcription CSV.\\\")\\n\",\"df=pd.read_csv(CSV_PATH)\\n\",\"df=df[[IMAGE_COL,TEXT_COL]].dropna().copy()\\n\",\"df.columns=[\\\"image_path\\\",\\\"text\\\"]\\n\",\"def resolve(p):\\n\",\"    p=Path(str(p))\\n\",\"    if p.is_file(): return str(p)\\n\",\"    for root in Path(\\\"/kaggle/input\\\").glob(\\\"*\\\"):\\n\",\"        q=root/p\\n\",\"        if q.is_file(): return str(q)\\n\",\"        hit=next(root.rglob(p.name),None)\\n\",\"        if hit: return str(hit)\\n\",\"    return None\\n\",\"df[\\\"image_path\\\"]=df.image_path.apply(resolve)\\n\",\"df[\\\"text\\\"]=df.text.astype(str)\\n\",\"df=df.dropna(subset=[\\\"image_path\\\"])\\n\",\"df=df[df.text.str.len()>0].reset_index(drop=True)\\n\",\"print(\\\"Usable line pairs:\\\",len(df)); display(df.head())\\n\",\"if len(df)<100: raise ValueError(\\\"Need at least 100 correctly labelled line images for a meaningful first experiment.\\\")\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"from sklearn.model_selection import train_test_split\\n\",\"train_df,val_df=train_test_split(df,test_size=0.1,random_state=42)\\n\",\"# Character vocabulary is learned only from training transcripts.\\n\",\"chars=sorted(set(\\\"\\\".join(train_df.text.tolist())))\\n\",\"char_to_id={c:i+1 for i,c in enumerate(chars)} # 0 is CTC blank\\n\",\"id_to_char={i:c for c,i in char_to_id.items()}\\n\",\"NUM_CLASSES=len(chars)+1\\n\",\"print(\\\"Unique characters:\\\",len(chars))\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"HEIGHT=64\\n\",\"MAX_WIDTH=1600\\n\",\"def load_line(path):\\n\",\"    im=Image.open(path).convert(\\\"L\\\")\\n\",\"    w=max(4,int(round(im.width*HEIGHT/im.height)))\\n\",\"    w=min(w,MAX_WIDTH)\\n\",\"    w=max(4,(w//4)*4)\\n\",\"    im=im.resize((w,HEIGHT),Image.Resampling.LANCZOS)\\n\",\"    arr=np.asarray(im,dtype=np.float32)/255.0\\n\",\"    return torch.from_numpy(arr).unsqueeze(0),w\\n\",\"class LineDataset(Dataset):\\n\",\"    def __init__(self,frame): self.frame=frame.reset_index(drop=True)\\n\",\"    def __len__(self): return len(self.frame)\\n\",\"    def __getitem__(self,i):\\n\",\"        row=self.frame.iloc[i]\\n\",\"        image,w=load_line(row.image_path)\\n\",\"        target=[char_to_id[c] for c in row.text if c in char_to_id]\\n\",\"        return image,torch.tensor(target,dtype=torch.long),w,row.text\\n\",\"def collate(batch):\\n\",\"    images,targets,widths,texts=zip(*batch)\\n\",\"    maxw=max(widths)\\n\",\"    padded=torch.zeros(len(images),1,HEIGHT,maxw)\\n\",\"    for i,x in enumerate(images): padded[i,:,:,:x.shape[-1]]=x\\n\",\"    target_lengths=torch.tensor([len(x) for x in targets],dtype=torch.long)\\n\",\"    flat=torch.cat(targets) if target_lengths.sum()>0 else torch.empty(0,dtype=torch.long)\\n\",\"    return padded,flat,target_lengths,torch.tensor([w//4 for w in widths]),list(texts)\\n\",\"train_ds=LineDataset(train_df); val_ds=LineDataset(val_df)\\n\",\"train_loader=DataLoader(train_ds,batch_size=8,shuffle=True,num_workers=2,collate_fn=collate)\\n\",\"val_loader=DataLoader(val_ds,batch_size=8,shuffle=False,num_workers=2,collate_fn=collate)\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"class CRNN(nn.Module):\\n\",\"    def __init__(self,nclass):\\n\",\"        super().__init__()\\n\",\"        self.cnn=nn.Sequential(\\n\",\"            nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2,2),\\n\",\"            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2,2),\\n\",\"            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU()\\n\",\"        )\\n\",\"        self.proj=nn.Linear(128*(HEIGHT//4),256)\\n\",\"        self.rnn=nn.LSTM(256,192,bidirectional=True,batch_first=True)\\n\",\"        self.head=nn.Linear(384,nclass)\\n\",\"    def forward(self,x):\\n\",\"        z=self.cnn(x)              # B,C,H,W/4\\n\",\"        z=z.permute(0,3,1,2).flatten(2)\\n\",\"        z=self.proj(z)\\n\",\"        z,_=self.rnn(z)\\n\",\"        return self.head(z).log_softmax(2)\\n\",\"device=\\\"cuda\\\" if torch.cuda.is_available() else \\\"cpu\\\"\\n\",\"model=CRNN(NUM_CLASSES).to(device)\\n\",\"criterion=nn.CTCLoss(blank=0,zero_infinity=True)\\n\",\"optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3)\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"def greedy_decode(log_probs):\\n\",\"    ids=log_probs.argmax(2).tolist()\\n\",\"    result=[]\\n\",\"    for seq in ids:\\n\",\"        out=[]; prev=0\\n\",\"        for idx in seq:\\n\",\"            if idx!=0 and idx!=prev: out.append(id_to_char.get(idx,\\\"\\\"))\\n\",\"            prev=idx\\n\",\"        result.append(\\\"\\\".join(out))\\n\",\"    return result\\n\",\"def cer_one(ref,hyp):\\n\",\"    # Character error rate = Levenshtein distance / reference length.\\n\",\"    a=list(ref); b=list(hyp); prev=list(range(len(b)+1))\\n\",\"    for i,ca in enumerate(a,1):\\n\",\"        cur=[i]\\n\",\"        for j,cb in enumerate(b,1):\\n\",\"            cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(ca!=cb)))\\n\",\"        prev=cur\\n\",\"    return prev[-1]/max(1,len(a))\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"# Initial baseline training. If GPU memory is limited, reduce batch_size above to 4.\\n\",\"EPOCHS=10\\n\",\"best_cer=999.0\\n\",\"for epoch in range(EPOCHS):\\n\",\"    model.train(); losses=[]\\n\",\"    for images,targets,tgt_lens,in_lens,texts in train_loader:\\n\",\"        images=images.to(device); targets=targets.to(device)\\n\",\"        optimizer.zero_grad()\\n\",\"        logp=model(images).permute(1,0,2)\\n\",\"        loss=criterion(logp,targets,in_lens.to(device),tgt_lens.to(device))\\n\",\"        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); optimizer.step()\\n\",\"        losses.append(float(loss.item()))\\n\",\"    model.eval(); refs=[]; hyps=[]\\n\",\"    with torch.no_grad():\\n\",\"        for images,targets,tgt_lens,in_lens,texts in val_loader:\\n\",\"            pred=greedy_decode(model(images.to(device)))\\n\",\"            refs.extend(texts); hyps.extend(pred)\\n\",\"    val_cer=float(np.mean([cer_one(r,h) for r,h in zip(refs,hyps)]))\\n\",\"    print(f\\\"Epoch {epoch+1}/{EPOCHS} | loss={np.mean(losses):.4f} | val_CER={val_cer:.4f}\\\")\\n\",\"    if val_cer<best_cer:\\n\",\"        best_cer=val_cer\\n\",\"        torch.save({\\\"state_dict\\\":model.state_dict(),\\\"chars\\\":chars,\\\"height\\\":HEIGHT}, \\\"/kaggle/working/noorix_urdu_crnn_best.pth\\\")\\n\",\"print(\\\"Best validation CER:\\\",best_cer)\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"# Save the Urdu recognition model, Unicode vocabulary, metadata and ZIP.\\n\",\"out=Path(\\\"/kaggle/working/NOORIX_Urdu_Line_OCR_Muhammad_Farhan\\\"); out.mkdir(exist_ok=True)\\n\",\"ckpt=torch.load(\\\"/kaggle/working/noorix_urdu_crnn_best.pth\\\",map_location=\\\"cpu\\\",weights_only=False)\\n\",\"torch.save(ckpt,out/\\\"noorix_urdu_crnn_best.pth\\\")\\n\",\"(out/\\\"vocabulary.json\\\").write_text(json.dumps({\\\"char_to_id\\\":char_to_id,\\\"id_to_char\\\":{str(k):v for k,v in id_to_char.items()}},ensure_ascii=False,indent=2),encoding=\\\"utf-8\\\")\\n\",\"(out/\\\"metadata.json\\\").write_text(json.dumps({\\\"project\\\":\\\"NOORIX-AI\\\",\\\"prepared_by\\\":\\\"Muhammad Farhan\\\",\\n\",\"\\\"architecture\\\":\\\"CRNN-BiLSTM-CTC\\\",\\\"script\\\":\\\"Urdu Unicode\\\",\\\"image_height\\\":HEIGHT,\\\"max_width\\\":MAX_WIDTH,\\n\",\"\\\"validation_CER\\\":best_cer,\\\"input\\\":\\\"grayscale line image; page segmentation is not included\\\"},ensure_ascii=False,indent=2),encoding=\\\"utf-8\\\")\\n\",\"z=shutil.make_archive(\\\"/kaggle/working/NOORIX_Urdu_Line_OCR_Muhammad_Farhan\\\",\\\"zip\\\",out)\\n\",\"print(\\\"ZIP:\\\",z)\\n\",\"\"]}],\"metadata\":{\"kernelspec\":{\"display_name\":\"Python 3\",\"language\":\"python\",\"name\":\"python3\"},\"language_info\":{\"name\":\"python\",\"version\":\"3\"}},\"nbformat\":4,\"nbformat_minor\":5},\"notebooks/speech/whisper_urdu_english_asr_muhammad_farhan.ipynb\":{\"cells\":[{\"cell_type\":\"markdown\",\"metadata\":{},\"source\":[\"# NOORIX-AI Urdu / English Speech-to-Text — Whisper Fine-tuning\\n\",\"**Prepared by Muhammad Farhan**\\n\",\"\\n\",\"Fine-tune a multilingual Whisper checkpoint for one language per run. This is an ASR notebook (speech → text), not text-to-speech.\\n\",\"\\n\",\"## Kaggle notebook title\\n\",\"`NOORIX-ASR-Whisper-Urdu-English-Muhammad-Farhan`\\n\",\"\\n\",\"## Dataset to add\\n\",\"- Urdu: Search Kaggle for `Mozilla Common Voice Urdu` or `Common Voice Corpus Urdu audio`. An alternative official dataset source is Mozilla Data Collective's Common Voice Urdu release; it contains audio clips and TSV transcriptions. Check its current access and terms.\\n\",\"- English: Search Kaggle for `Common Voice English`.\\n\",\"- Your own recordings: use a CSV with `audio_path,text` columns. Audio must be clear, consented speech, and transcription must match exactly.\\n\",\"\\n\",\"Choose `LANGUAGE=\\\"urdu\\\"` or `LANGUAGE=\\\"english\\\"` in the config cell. Do not mix languages in one run without a deliberate multilingual data strategy. Whisper is large; use a Kaggle GPU and begin with a small subset.\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"!pip -q install -U transformers datasets[audio] evaluate jiwer accelerate soundfile scikit-learn\",\"\",\"import os, glob, json, shutil\",\"\",\"from pathlib import Path\",\"\",\"import pandas as pd, numpy as np, torch\",\"\",\"from datasets import Dataset, Audio\",\"\",\"from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments\",\"\",\"import evaluate\",\"\",\"print(\\\"GPU:\\\",torch.cuda.get_device_name(0) if torch.cuda.is_available() else \\\"CPU\\\")\",\"\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"# Attach a Kaggle dataset containing audio and a transcription CSV.\",\"for p in glob.glob('/kaggle/input/*'): print(p)\",\"CSV_PATH = '' # injected by the master notebook\",\"AUDIO_COL = 'audio_path'\",\"TEXT_COL = 'text'\",\"LANGUAGE = 'urdu' # set to 'english' for English run\",\"MODEL_NAME = 'openai/whisper-small'\",\"if not CSV_PATH: raise ValueError('Set CSV_PATH to your labelled audio manifest.')\",\"sep = '\\\\t' if CSV_PATH.lower().endswith('.tsv') else ','\",\"df = pd.read_csv(CSV_PATH, sep=sep)\",\"df = df[[AUDIO_COL, TEXT_COL] + (['client_id'] if 'client_id' in df.columns else [])].dropna(subset=[AUDIO_COL, TEXT_COL])\",\"df.columns = ['audio_path', 'text'] + (['client_id'] if 'client_id' in df.columns else [])\",\"def resolve(p):\",\"    p = Path(str(p))\",\"    if p.is_file(): return str(p)\",\"    for root in Path('/kaggle/input').glob('*'):\",\"        q = root / p\",\"        if q.is_file(): return str(q)\",\"        hits = list(root.rglob(p.name))\",\"        if hits: return str(hits[0])\",\"    return None\",\"df['audio_path'] = df.audio_path.apply(resolve)\",\"df['text'] = df['text'].astype(str).str.strip()\",\"df = df.dropna(subset=['audio_path'])\",\"df = df[df['text'].ne('')].drop_duplicates(subset=['audio_path']).reset_index(drop=True)\",\"print('Usable clips:', len(df)); display(df.head())\",\"if len(df) < 100: raise ValueError('Need at least 100 labelled audio rows before fine-tuning.')\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"# Speaker-disjoint train/validation split when speaker IDs are available.\",\"MAX_SAMPLES = 5000\",\"if len(df) > MAX_SAMPLES:\",\"    df = df.sample(MAX_SAMPLES, random_state=42).reset_index(drop=True)\",\"if 'client_id' in df.columns and df['client_id'].nunique() >= 2:\",\"    from sklearn.model_selection import GroupShuffleSplit\",\"    splitter = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)\",\"    train_idx, val_idx = next(splitter.split(df, groups=df['client_id'].fillna('unknown')))\",\"    train_df = df.iloc[train_idx].drop(columns=['client_id'], errors='ignore').reset_index(drop=True)\",\"    val_df = df.iloc[val_idx].drop(columns=['client_id'], errors='ignore').reset_index(drop=True)\",\"    print('Speaker-disjoint split enabled; train speakers:', df.iloc[train_idx]['client_id'].nunique(), 'validation speakers:', df.iloc[val_idx]['client_id'].nunique())\",\"else:\",\"    from sklearn.model_selection import train_test_split\",\"    train_df, val_df = train_test_split(df.drop(columns=['client_id'], errors='ignore'), test_size=0.1, random_state=42)\",\"    train_df = train_df.reset_index(drop=True); val_df = val_df.reset_index(drop=True)\",\"train_ds = Dataset.from_pandas(train_df)\",\"val_ds = Dataset.from_pandas(val_df)\",\"for ds_name, ds_value in [('train', train_ds), ('validation', val_ds)]:\",\"    ds_value = ds_value.rename_column('audio_path', 'audio')\",\"    ds_value = ds_value.cast_column('audio', Audio(sampling_rate=16000))\",\"    if ds_name == 'train': train_ds = ds_value\",\"    else: val_ds = ds_value\",\"processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task='transcribe')\",\"model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)\",\"model.generation_config.language = LANGUAGE\",\"model.generation_config.task = 'transcribe'\",\"model.generation_config.forced_decoder_ids = None\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"def prepare(batch):\\n\",\"    audio=batch[\\\"audio\\\"]\\n\",\"    batch[\\\"input_features\\\"]=processor.feature_extractor(audio[\\\"array\\\"],sampling_rate=audio[\\\"sampling_rate\\\"]).input_features[0]\\n\",\"    batch[\\\"labels\\\"]=processor.tokenizer(batch[\\\"text\\\"]).input_ids\\n\",\"    return batch\\n\",\"train_ds=splits[\\\"train\\\"].map(prepare,remove_columns=splits[\\\"train\\\"].column_names)\\n\",\"val_ds=splits[\\\"test\\\"].map(prepare,remove_columns=splits[\\\"test\\\"].column_names)\\n\",\"from dataclasses import dataclass\\n\",\"@dataclass\\n\",\"class DataCollator:\\n\",\"    processor: object\\n\",\"    def __call__(self,features):\\n\",\"        inputs=[{\\\"input_features\\\":f[\\\"input_features\\\"]} for f in features]\\n\",\"        batch=self.processor.feature_extractor.pad(inputs,return_tensors=\\\"pt\\\")\\n\",\"        labels=[{\\\"input_ids\\\":f[\\\"labels\\\"]} for f in features]\\n\",\"        lab=self.processor.tokenizer.pad(labels,return_tensors=\\\"pt\\\")\\n\",\"        batch[\\\"labels\\\"]=lab[\\\"input_ids\\\"].masked_fill(lab.attention_mask.ne(1),-100)\\n\",\"        return batch\\n\",\"collator=DataCollator(processor)\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"cer=evaluate.load(\\\"cer\\\"); wer=evaluate.load(\\\"wer\\\")\\n\",\"def metrics_fn(pred):\\n\",\"    pred_ids=pred.predictions[0] if isinstance(pred.predictions,tuple) else pred.predictions\\n\",\"    label_ids=pred.label_ids.copy(); label_ids[label_ids==-100]=processor.tokenizer.pad_token_id\\n\",\"    p=processor.tokenizer.batch_decode(pred_ids,skip_special_tokens=True)\\n\",\"    r=processor.tokenizer.batch_decode(label_ids,skip_special_tokens=True)\\n\",\"    return {\\\"cer\\\":cer.compute(predictions=p,references=r),\\\"wer\\\":wer.compute(predictions=p,references=r)}\\n\",\"args=Seq2SeqTrainingArguments(output_dir=\\\"/kaggle/working/noorix_asr_ckpt\\\",per_device_train_batch_size=4,\\n\",\"    per_device_eval_batch_size=4,gradient_accumulation_steps=2,learning_rate=1e-5,num_train_epochs=2,\\n\",\"    fp16=torch.cuda.is_available(),eval_strategy=\\\"epoch\\\",save_strategy=\\\"epoch\\\",predict_with_generate=True,\\n\",\"    generation_max_length=128,logging_steps=25,save_total_limit=2,load_best_model_at_end=True,\\n\",\"    metric_for_best_model=\\\"cer\\\",greater_is_better=False,report_to=\\\"none\\\")\\n\",\"trainer=Seq2SeqTrainer(model=model,args=args,train_dataset=train_ds,eval_dataset=val_ds,\\n\",\"    data_collator=collator,compute_metrics=metrics_fn,processing_class=processor)\\n\",\"trainer.train()\\n\",\"\"]},{\"cell_type\":\"code\",\"metadata\":{},\"execution_count\":null,\"outputs\":[],\"source\":[\"metrics=trainer.evaluate(); print(metrics)\\n\",\"out=Path(\\\"/kaggle/working/NOORIX_ASR_Whisper_\\\"+LANGUAGE+\\\"_Muhammad_Farhan\\\"); out.mkdir(exist_ok=True)\\n\",\"model.save_pretrained(out/\\\"model\\\"); processor.save_pretrained(out/\\\"processor\\\")\\n\",\"(out/\\\"metadata.json\\\").write_text(json.dumps({\\\"project\\\":\\\"NOORIX-AI\\\",\\\"prepared_by\\\":\\\"Muhammad Farhan\\\",\\n\",\"\\\"task\\\":\\\"ASR\\\",\\\"language\\\":LANGUAGE,\\\"base_model\\\":MODEL_NAME,\\\"validation_metrics\\\":{k:float(v) for k,v in metrics.items() if isinstance(v,(int,float))},\\n\",\"\\\"note\\\":\\\"Validate on a separate real-world test set; local CPU latency and memory must be measured.\\\"},indent=2),encoding=\\\"utf-8\\\")\\n\",\"z=shutil.make_archive(\\\"/kaggle/working/NOORIX_ASR_Whisper_\\\"+LANGUAGE+\\\"_Muhammad_Farhan\\\",\\\"zip\\\",out)\\n\",\"print(\\\"ZIP:\\\",z)\\n\",\"\"]}],\"metadata\":{\"kernelspec\":{\"display_name\":\"Python 3\",\"language\":\"python\",\"name\":\"python3\"},\"language_info\":{\"name\":\"python\",\"version\":\"3\"}},\"nbformat\":4,\"nbformat_minor\":5}}"EMBEDDED = json.loads(EMBEDDED_JSON)for relative_path, notebook_data in EMBEDDED.items():    destination = REPO / relative_path    destination.parent.mkdir(parents=True, exist_ok=True)    destination.write_text(json.dumps(notebook_data, ensure_ascii=False), encoding='utf-8')print('Built-in specialist notebooks installed:', len(EMBEDDED))print('No GitHub clone, access token, or private-repository login is required.')NOTEBOOKS = { 'printed_english_ocr': REPO/'notebooks/ocr/printed_ocr_training_muhammad_farhan.ipynb', 'handwritten_english_ocr': REPO/'notebooks/ocr/handwritten_ocr_training_muhammad_farhan.ipynb', 'urdu_line_ocr': REPO/'notebooks/ocr/urdu_printed_handwritten_crnn_ctc_muhammad_farhan.ipynb', 'pkr_currency': REPO/'notebooks/currency/pkr_mobilenetv2_training_muhammad_farhan.ipynb', 'yolo_objects_person': REPO/'notebooks/detection/yolov8n_objects_person_muhammad_farhan.ipynb', 'whisper_urdu_asr': REPO/'notebooks/speech/whisper_urdu_english_asr_muhammad_farhan.ipynb', 'whisper_english_asr': REPO/'notebooks/speech/whisper_urdu_english_asr_muhammad_farhan.ipynb',}print('Notebook paths configured:', list(NOTEBOOKS))

## 3. Run enabled specialist stages
Each stage runs in an isolated notebook kernel. Failures are recorded and do not silently become trained models. The baseline profile is intentionally limited; check metrics and runtime before increasing training.

In [ ]:
!pip -q install nbclient nbformat
import re, json, time, traceback, shutil, zipfile, hashlib
import nbformat
from nbclient import NotebookClient

STAGE_ARCHIVE_PREFIXES = {
    'printed_english_ocr': 'NOORIX_PRINTED_OCR',
    'handwritten_english_ocr': 'NOORIX_HANDWRITTEN_OCR',
    'urdu_line_ocr': 'NOORIX_Urdu_Line_OCR',
    'pkr_currency': 'NOORIX_PKR_MobileNetV2',
    'yolo_objects_person': 'NOORIX_Object_Person_YOLOv8n',
    'whisper_urdu_asr': 'NOORIX_ASR_Whisper_urdu',
    'whisper_english_asr': 'NOORIX_ASR_Whisper_english',
}

def set_assignment(nb, variable, value):
    # Works with normal assignments and compact semicolon-separated source cells.
    pattern = re.compile(rf"""(?<![A-Za-z0-9_])({re.escape(variable)}\s*=\s*)(?:"[^"]*"|'[^']*'|[^;\n#]+)""")
    replacement_value = json.dumps(value, ensure_ascii=False)
    for cell in nb.cells:
        if cell.cell_type != 'code':
            continue
        source = cell.source
        updated, count = pattern.subn(
            lambda match: match.group(1) + replacement_value,
            source,
            count=1
        )
        if count:
            cell.source = updated
            return
    raise RuntimeError(f'Could not locate assignment for {variable!r} in notebook.')

def run_stage(name, cfg):
    src = NOTEBOOKS[name]
    if not src.is_file():
        raise FileNotFoundError(src)

    error_file = OUT / f'{name}_ERROR.txt'
    if error_file.exists():
        error_file.unlink()

    for stale in WORK.glob(STAGE_ARCHIVE_PREFIXES[name] + '*.zip'):
        if stale.is_file():
            stale.unlink()

    nb = nbformat.read(src, as_version=4)

    if name in ('printed_english_ocr', 'handwritten_english_ocr'):
        set_assignment(nb, 'ANNOTATION_CSV', cfg['annotation_csv'])
        set_assignment(nb, 'IMAGE_COL', cfg.get('image_col', ''))
        set_assignment(nb, 'TEXT_COL', cfg.get('text_col', ''))
    elif name == 'urdu_line_ocr':
        set_assignment(nb, 'CSV_PATH', cfg['csv_path'])
        set_assignment(nb, 'IMAGE_COL', cfg.get('image_col', 'image_path'))
        set_assignment(nb, 'TEXT_COL', cfg.get('text_col', 'text'))
    elif name == 'pkr_currency':
        set_assignment(nb, 'DATA_DIR', cfg['data_dir'])
    elif name == 'yolo_objects_person':
        set_assignment(nb, 'CUSTOM_DATA_YAML', cfg.get('custom_data_yaml', ''))
        set_assignment(nb, 'TEST_IMAGE', cfg.get('test_image', ''))
    elif name in ('whisper_urdu_asr', 'whisper_english_asr'):
        set_assignment(nb, 'CSV_PATH', cfg['csv_path'])
        set_assignment(nb, 'AUDIO_COL', cfg.get('audio_col', 'audio_path'))
        set_assignment(nb, 'TEXT_COL', cfg.get('text_col', 'text'))
        set_assignment(nb, 'LANGUAGE', cfg['language'])

    temp = OUT / f'{name}_executed.ipynb'
    client = NotebookClient(
        nb,
        timeout=MAX_NOTEBOOK_TIMEOUT,
        kernel_name='python3',
        allow_errors=False,
        resources={'metadata': {'path': str(WORK)}}
    )
    executed = client.execute()
    nbformat.write(executed, temp)
    return temp

results = []
for name, cfg in CONFIG.items():
    if not cfg['run']:
        continue
    print('\n' + '=' * 72 + '\nSTART:', name)
    start = time.time()
    try:
        executed = run_stage(name, cfg)
        results.append({
            'stage': name,
            'status': 'success',
            'seconds': round(time.time() - start, 1),
            'executed_notebook': str(executed)
        })
        print('STAGE SUCCESS:', name)
    except Exception as exc:
        details = traceback.format_exc()
        (OUT / f'{name}_ERROR.txt').write_text(details, encoding='utf-8')
        results.append({
            'stage': name,
            'status': 'failed',
            'seconds': round(time.time() - start, 1),
            'error': str(exc)
        })
        print('STAGE FAILED; continuing:', name, '|', repr(exc))

print('\nStage summary:', json.dumps(results, indent=2, ensure_ascii=False))


## 4. Build ONE organized model ZIP
The stage ZIPs are temporary outputs only. The final package extracts each stage into models/<task>/ and contains no nested ZIP files.

In [ ]:
# Extract successful stage archives into separate model folders and create one final ZIP.
import zipfile, hashlib, json, shutil, time

MODELS_DIR = OUT / 'models'
if MODELS_DIR.exists():
    shutil.rmtree(MODELS_DIR)
MODELS_DIR.mkdir(parents=True)

for item in results:
    if item['status'] != 'success':
        continue
    stage = item['stage']
    prefix = STAGE_ARCHIVE_PREFIXES[stage]
    archives = [p for p in WORK.glob(prefix + '*.zip') if p.is_file()]
    if not archives:
        item['status'] = 'success_no_archive_found'
        continue
    archive = max(archives, key=lambda p: p.stat().st_mtime)
    target = MODELS_DIR / stage
    target.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(archive, 'r') as zf:
            bad = zf.testzip()
            if bad:
                raise RuntimeError('Corrupt stage archive member: ' + str(bad))
            for info in zf.infolist():
                dest = (target / info.filename).resolve()
                if target.resolve() not in dest.parents and dest != target.resolve():
                    raise RuntimeError('Unsafe archive path: ' + info.filename)
            zf.extractall(target)
        item['model_folder'] = 'models/' + stage + '/'
    except Exception as exc:
        item['status'] = 'package_failed'
        item['package_error'] = str(exc)
        shutil.rmtree(target, ignore_errors=True)

# Explicitly list disabled stages.
seen = {item['stage'] for item in results}
for stage, cfg in CONFIG.items():
    if stage not in seen:
        results.append({'stage':stage,'status':'skipped_not_trained',
                        'reason':'No verified labelled manifest/configuration or stage disabled.'})

manifest = {
    'project':'NOORIX-AI',
    'prepared_by':'Muhammad Farhan',
    'package_version':'unified-offline-model-suite-v2',
    'created_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
    'architecture':'One unified app/API with task-specific specialist models',
    'stages':results,
    'offline_inference':'Possible after model weights, dependencies and compatible task adapters are installed locally.',
    'warnings':[
        'PKR classification estimates denomination/side only; it does not authenticate notes.',
        'YOLOv8 needs its own decoder and is not plug-compatible with the current SSD adapter.',
        'ASR is speech-to-text; TTS is a separate task and is not trained here.',
        'OCR stages are skipped unless exact image/transcription labels are configured.',
        'Successful execution and ZIP creation do not prove accuracy or app integration.'
    ]
}
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT/'README_OFFLINE.md').write_text(
    '# NOORIX-AI Unified Model Package\\n\\n'
    'Each trained model is in models/<task>/. Read manifest.json for stage status and limitations. '
    'Install the matching runtime and NOORIX adapter before offline inference. YOLO ONNX requires a YOLO decoder. '
    'ASR does not provide TTS.\\n', encoding='utf-8'
)

checks=[]
for file in sorted(p for p in OUT.rglob('*') if p.is_file() and p.name != 'checksums.sha256'):
    checks.append(hashlib.sha256(file.read_bytes()).hexdigest() + '  ' + file.relative_to(OUT).as_posix())
(OUT/'checksums.sha256').write_text('\\n'.join(checks)+'\\n',encoding='utf-8')

combined = WORK / 'NOORIX_AI_ALL_MODELS.zip'
if combined.exists():
    combined.unlink()
with zipfile.ZipFile(combined,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(p for p in OUT.rglob('*') if p.is_file()):
        zf.write(file,arcname=file.relative_to(OUT).as_posix())
with zipfile.ZipFile(combined,'r') as zf:
    bad=zf.testzip()
    if bad:
        raise RuntimeError('Combined ZIP integrity failure: ' + str(bad))
    print('ZIP entries:',len(zf.namelist()))
    print('Top-level folders:',sorted({x.split('/')[0] for x in zf.namelist()}))
print('FINAL DOWNLOAD:',combined)
print('Size MB:',round(combined.stat().st_size/(1024*1024),2))
print('Model folders:',[p.name for p in MODELS_DIR.iterdir() if p.is_dir()])

## 5. Offline deployment notes
- YOLOv8 **can run offline** when its exported ONNX or .pt weights are included locally, the inference runtime is installed, and NOORIX uses a compatible YOLO decoder. The exported ONNX is included in the object-detection model folder.
- The existing NOORIX SSD adapter cannot directly decode YOLO outputs; a separate YOLO adapter and numerical tests are required.
- PKR classification estimates denomination/side only; it does not authenticate notes.
- Whisper ASR is speech-to-text, not TTS. TTS needs a separate offline voice model/runtime.
- OCR is skipped unless exact image-to-text annotations are configured.
- Kaggle training may require internet for packages and pretrained weights. Offline refers to inference after weights/dependencies are installed on the device.
- Windows EXE and Android APK need separate packaging, adapters, and device performance tests.
- A generated ZIP is not proof of model accuracy or successful integration.

In [ ]:
print(json.dumps(manifest,indent=2,ensure_ascii=False))
